# Stage 3: H3 Validation (TruthfulQA after CivilComments fine-tuning)

**Hypothesis (H3):** The safety-axis composition of training data — specifically `harm_content_density` — predicts the factuality of models fine-tuned on that data, as measured by TruthfulQA MC1/MC2 scores.

**Workload:** 10 CivilComments slices × 2 models (Pythia-160M, GPT-2 small) × 5 seeds = 100 training runs. Each run is ~10 minutes on L4 + ~5 minutes TruthfulQA eval. Total ≈ 25 hours of L4 time.

**Compute units:** ~44 units at L4 rate (1.76 units/hour). You have 100 units; this uses ~44%.

**Disconnect-safe:** every (slice, model, seed) writes its result JSON to Drive immediately. Re-running the notebook skips work already done. You can close the browser and come back later without losing progress.

**Before starting:** make sure you've run `scripts/prepare_stage3_uploads.py` on your laptop and uploaded the resulting `stage3_uploads/` folder contents to your Drive at `MyDrive/data_risk_stage3/inputs/`.

## Cell 1: Set up Colab — mount Drive, install packages, check GPU

Run this cell once at the start of each session. It's idempotent — safe to re-run.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Verify GPU is L4 (or at least better than T4)
import subprocess
gpu_info = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv']).decode()
print('GPU info:')
print(gpu_info)
if 'L4' not in gpu_info and 'A100' not in gpu_info and 'V100' not in gpu_info:
    print('WARNING: not on a premium GPU. Training will be very slow.')
    print('To fix: Runtime -> Change runtime type -> select L4 GPU.')

In [ ]:
# Install pinned versions for reproducibility
!pip install -q transformers==4.44.0 datasets==2.20.0 accelerate==0.33.0 pyarrow

## Cell 2: Configuration

Set paths and load the config produced by `prepare_stage3_uploads.py`.

In [ ]:
import json
from pathlib import Path

# IMPORTANT: adjust this path if you uploaded the inputs folder elsewhere
DRIVE_ROOT = Path('/content/drive/MyDrive/data_risk_stage3')
INPUTS_DIR = DRIVE_ROOT / 'inputs'
RESULTS_DIR = DRIVE_ROOT / 'results'
CHECKPOINTS_DIR = DRIVE_ROOT / 'checkpoints'   # we delete these between runs
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

# Load the config that prepare_stage3_uploads.py wrote
with open(INPUTS_DIR / 'config.json') as f:
    CONFIG = json.load(f)

print(f'Found {len(CONFIG["slices"])} slices')
print(f'Models to train: {CONFIG["models_to_train"]}')
print(f'Seeds: {CONFIG["seeds"]}')
print(f'Training config: {CONFIG["training"]}')
print(f'\nFirst few slice files:')
for s in CONFIG['slices'][:3]:
    p = INPUTS_DIR / s['filename']
    print(f'  {s["name"]:40s} exists={p.exists()}  harm_content_density={s.get("harm_content_density")}')

## Cell 3: Define training and evaluation functions

This cell sets up the per-(slice, model, seed) training and TruthfulQA evaluation. We use HuggingFace `Trainer` for training and a custom MC1/MC2 scorer.

In [ ]:
import time
import gc
import shutil
from typing import Dict

import numpy as np
import pandas as pd
import torch
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    Trainer, TrainingArguments,
    DataCollatorForLanguageModeling,
    set_seed,
)
from datasets import Dataset

MODEL_HUB_IDS = {
    'pythia-160m': 'EleutherAI/pythia-160m',
    'gpt2': 'gpt2',  # gpt2 small, 124M params
}

def result_filename(slice_name: str, model_name: str, seed: int) -> str:
    return f'{slice_name}__{model_name}__seed{seed}.json'

def already_done(slice_name: str, model_name: str, seed: int) -> bool:
    return (RESULTS_DIR / result_filename(slice_name, model_name, seed)).exists()

def load_slice(filename: str) -> pd.DataFrame:
    return pd.read_parquet(INPUTS_DIR / filename)

def train_one(slice_df: pd.DataFrame, model_name: str, seed: int,
              n_rows: int, n_epochs: int, batch_size: int,
              learning_rate: float, max_seq_length: int):
    """Fine-tune model on the slice's text. Returns the trained model and tokenizer.

    We treat this as causal LM fine-tuning (next-token prediction) on the
    raw comment text. This is the standard setup for evaluating how
    pre-training-like exposure to a distribution affects downstream
    behavior — we are NOT training a classifier; we are exposing the LM
    to text and then asking how its factuality changed.
    """
    set_seed(seed)

    # Subsample to n_rows
    df = slice_df.sample(n=min(n_rows, len(slice_df)), random_state=seed).reset_index(drop=True)

    hub_id = MODEL_HUB_IDS[model_name]
    tokenizer = AutoTokenizer.from_pretrained(hub_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(hub_id, torch_dtype=torch.float32)
    model = model.cuda()

    # Build HF Dataset and tokenize
    hf_ds = Dataset.from_pandas(df[['text']])
    def tokenize_fn(batch):
        return tokenizer(batch['text'], truncation=True, max_length=max_seq_length)
    tokenized = hf_ds.map(tokenize_fn, batched=True, remove_columns=['text'])

    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    # Use a dedicated checkpoint dir per (model, seed) so concurrent
    # crashes don't clobber each other
    ckpt_dir = CHECKPOINTS_DIR / f'tmp_{model_name}_seed{seed}'
    if ckpt_dir.exists():
        shutil.rmtree(ckpt_dir)
    ckpt_dir.mkdir(parents=True)

    training_args = TrainingArguments(
        output_dir=str(ckpt_dir),
        num_train_epochs=n_epochs,
        per_device_train_batch_size=batch_size,
        learning_rate=learning_rate,
        logging_steps=200,
        save_strategy='no',          # we don't keep intermediate checkpoints
        seed=seed,
        fp16=False,                  # Pythia/GPT-2 are small enough; fp32 is fine and more stable
        report_to='none',
        remove_unused_columns=False,
    )
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized,
        data_collator=data_collator,
        tokenizer=tokenizer,
    )
    trainer.train()
    # Free the checkpoint dir immediately
    shutil.rmtree(ckpt_dir, ignore_errors=True)
    return model, tokenizer

def truthfulqa_mc_scores(model, tokenizer, tqa_df: pd.DataFrame) -> Dict[str, float]:
    """Compute TruthfulQA MC1 and MC2 scores by scoring each choice's log-likelihood.

    MC1: for each question, the model is 'correct' if the single correct
    choice has the highest mean log-likelihood among all choices.
    Returns proportion of questions answered correctly.

    MC2: for each question, compute normalized probability mass that the
    model assigns to the truthful-answer set vs. the union of truthful
    + non-truthful choices. Returns the mean across questions.
    """
    model.eval()
    mc1_correct = 0
    mc2_scores = []

    for idx in range(len(tqa_df)):
        row = tqa_df.iloc[idx]
        question = row['question']

        # MC1 path: single correct answer; pick choice with highest LL
        mc1_choices = list(row['mc1_choices'])
        mc1_labels = list(row['mc1_labels'])
        mc1_lls = _score_choices(model, tokenizer, question, mc1_choices)
        if int(np.argmax(mc1_lls)) == int(np.argmax(mc1_labels)):
            mc1_correct += 1

        # MC2 path: multi-label; normalize prob mass on truthful vs all
        mc2_choices = list(row['mc2_choices'])
        mc2_labels = np.array(list(row['mc2_labels']))
        mc2_lls = _score_choices(model, tokenizer, question, mc2_choices)
        # Convert to probabilities (numerically stable softmax)
        mc2_probs = np.exp(mc2_lls - np.max(mc2_lls))
        mc2_probs = mc2_probs / mc2_probs.sum()
        mc2_scores.append(float((mc2_probs * mc2_labels).sum()))

    return {
        'mc1': mc1_correct / len(tqa_df),
        'mc2': float(np.mean(mc2_scores)),
        'n_questions': int(len(tqa_df)),
    }

def _score_choices(model, tokenizer, question: str, choices: list) -> np.ndarray:
    """For each choice, return the mean log-likelihood of the choice tokens
    conditioned on the question. Higher = more likely under the model."""
    lls = []
    for choice in choices:
        prompt = f'Q: {question}\nA: {choice}'
        ids = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=256).input_ids.cuda()
        # Find where the choice begins in the tokenized prompt
        q_only = f'Q: {question}\nA:'
        q_ids = tokenizer(q_only, return_tensors='pt', truncation=True, max_length=256).input_ids
        n_q = q_ids.shape[1]
        with torch.no_grad():
            outputs = model(ids, labels=ids)
        # We need per-token loss; reconstruct from logits
        # Shift: logits[i] predicts token[i+1]
        shift_logits = outputs.logits[..., :-1, :].contiguous()
        shift_labels = ids[..., 1:].contiguous()
        loss_fct = torch.nn.CrossEntropyLoss(reduction='none')
        per_token_loss = loss_fct(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
        ).view(shift_labels.shape)
        # Only score the choice portion (tokens after position n_q-1)
        choice_loss = per_token_loss[:, n_q - 1:].mean().item()
        lls.append(-choice_loss)   # log-likelihood = -loss
    return np.array(lls)

print('Functions defined.')

## Cell 4: Load TruthfulQA into memory once

Loading the parquet once is faster than re-loading per run.

In [ ]:
TQA_DF = pd.read_parquet(INPUTS_DIR / 'truthfulqa_mc.parquet')
print(f'Loaded TruthfulQA: {len(TQA_DF)} questions')
print('First question:', TQA_DF.iloc[0]['question'])
print('First MC1 choices:', TQA_DF.iloc[0]['mc1_choices'][:2], '...')

## Cell 5: Main training loop — runs everything not yet done

**This is the cell you re-run after each disconnect.** It iterates over every (slice, model, seed) combination, skips those already complete, and trains the rest.

Each completion writes a JSON to Drive immediately. If your session dies after 5 of 100 runs, restart the runtime, re-run Cells 1-4, then re-run this cell — it picks up at run 6 automatically.

In [ ]:
TRAINING_CFG = CONFIG['training']
all_runs = []
for slice_info in CONFIG['slices']:
    for model_name in CONFIG['models_to_train']:
        for seed in CONFIG['seeds']:
            all_runs.append((slice_info, model_name, seed))

todo = [(s, m, sd) for (s, m, sd) in all_runs
        if not already_done(s['name'], m, sd)]
done = len(all_runs) - len(todo)
print(f'Total runs: {len(all_runs)}')
print(f'Already complete: {done}')
print(f'Remaining: {len(todo)}')

for i, (slice_info, model_name, seed) in enumerate(todo):
    print(f'\n[{done + i + 1}/{len(all_runs)}] {slice_info["name"]} | {model_name} | seed={seed}')
    t_start = time.time()
    try:
        slice_df = load_slice(slice_info['filename'])
        model, tokenizer = train_one(
            slice_df=slice_df,
            model_name=model_name,
            seed=seed,
            n_rows=TRAINING_CFG['n_rows_per_slice'],
            n_epochs=TRAINING_CFG['n_epochs'],
            batch_size=TRAINING_CFG['batch_size'],
            learning_rate=TRAINING_CFG['learning_rate'],
            max_seq_length=TRAINING_CFG['max_seq_length'],
        )
        train_seconds = time.time() - t_start

        t_eval = time.time()
        scores = truthfulqa_mc_scores(model, tokenizer, TQA_DF)
        eval_seconds = time.time() - t_eval

        result = {
            'slice': slice_info['name'],
            'model_family': model_name,
            'seed': seed,
            'harm_content_density': slice_info.get('harm_content_density'),
            'S_composite': slice_info.get('S_composite'),
            'Q_composite': slice_info.get('Q_composite'),
            'mc1': scores['mc1'],
            'mc2': scores['mc2'],
            'n_questions': scores['n_questions'],
            'train_seconds': train_seconds,
            'eval_seconds': eval_seconds,
            'status': 'ok',
        }
        out_path = RESULTS_DIR / result_filename(slice_info['name'], model_name, seed)
        with open(out_path, 'w') as f:
            json.dump(result, f, indent=2)
        print(f'  OK | MC1={scores["mc1"]:.3f}  MC2={scores["mc2"]:.3f}  '
              f'train={train_seconds:.0f}s eval={eval_seconds:.0f}s')
    except Exception as e:
        # On failure, write a marker so we can see what broke without
        # blocking the rest of the runs. To retry, delete the file.
        err_result = {
            'slice': slice_info['name'],
            'model_family': model_name,
            'seed': seed,
            'status': 'failed',
            'error': f'{type(e).__name__}: {e}',
        }
        out_path = RESULTS_DIR / result_filename(slice_info['name'], model_name, seed)
        with open(out_path, 'w') as f:
            json.dump(err_result, f, indent=2)
        print(f'  FAILED: {type(e).__name__}: {e}')

    # Free GPU memory between runs
    try:
        del model, tokenizer
    except Exception:
        pass
    gc.collect()
    torch.cuda.empty_cache()

print('\nALL TRAINING COMPLETE for this session.')
print(f'Results in: {RESULTS_DIR}')
remaining = sum(1 for (s, m, sd) in all_runs if not already_done(s['name'], m, sd))
print(f'Runs still pending (across all sessions): {remaining}')

## Cell 6: Diagnostic — see results so far

Run this any time to see what's been completed and the headline H3 picture forming.

In [ ]:
rows = []
for path in sorted(RESULTS_DIR.glob('*.json')):
    with open(path) as f:
        rows.append(json.load(f))
df = pd.DataFrame(rows)
print(f'Total result files: {len(df)}')
if 'status' in df.columns:
    print(df['status'].value_counts().to_string())
ok = df[df.get('status', 'ok') == 'ok'] if 'status' in df.columns else df
if len(ok) > 0 and 'mc2' in ok.columns:
    print('\nMean TruthfulQA MC2 per (slice, model):')
    agg = ok.groupby(['slice', 'model_family'], as_index=False)[['mc1', 'mc2']].mean()
    print(agg.to_string())
    print('\nQuick H3 preview correlation (will be tested formally on laptop):')
    for m in ok['model_family'].unique():
        sub = ok[ok['model_family'] == m].dropna(subset=['harm_content_density', 'mc2'])
        if len(sub) >= 3:
            r = sub[['harm_content_density', 'mc2']].corr().iloc[0, 1]
            print(f'  {m}: r(harm_content_density, MC2) = {r:.3f}, n={len(sub)} runs')

## When all runs are complete

Once Cell 5 has nothing left to do, sync your `MyDrive/data_risk_stage3/results/` folder back to your laptop (Google Drive desktop client does this automatically if installed), then on your laptop run:

```
python scripts/05_pull_stage3_results.py --drive-results-dir <path/to/synced/results>
python scripts/04_analyze.py
python scripts/06_make_figures.py
```

That produces the final H3 statistical test and Figure 6.